<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione3/Lezione3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 3 — Conversazione & Memoria

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Martedì 26/05/2026

---

### 🎯 Obiettivi
- ✅ Gestire la conversation history (multi-turno)
- ✅ Implementare troncamento e sliding window
- ✅ Aggiungere lo streaming delle risposte
- ✅ Salvare e ricaricare la memoria su file JSON

In [ ]:
# Setup
!pip install anthropic -q
from google.colab import userdata
import anthropic, os, json

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 8.7 MB/s eta 0:00:00
✅ Setup completato!


---
## 1. Conversation History — il chatbot che ricorda

Il modello **non ha memoria propria**. Per farlo 'ricordare', dobbiamo inviargli tutta la conversazione ad ogni chiamata.

In [ ]:
# Chatbot multi-turno base
history = []

def chat(messaggio, system=None):
    """Invia un messaggio mantenendo la history."""
    # 1. Aggiungi il messaggio dell'utente
    history.append({"role": "user", "content": messaggio})

    # 2. Invia TUTTA la history al modello
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 500,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text

    # 3. Aggiungi la risposta alla history
    history.append({"role": "assistant", "content": testo})

    return testo

print("✅ Funzione chat pronta!")

✅ Funzione chat pronta!


In [ ]:
# Proviamo la memoria!
history = []  # Reset

print("👤 Mi chiamo Marco e sono di Sassari.")
r1 = chat("Mi chiamo Marco e sono di Sassari.")
print(f"🤖 {r1}\n")

print("👤 Qual è la capitale della Sardegna?")
r2 = chat("Qual è la capitale della Sardegna?")
print(f"🤖 {r2}\n")

print("👤 Come mi chiamo?")
r3 = chat("Come mi chiamo?")  # Ricorda il nome?
print(f"🤖 {r3}\n")

print(f"📊 Messaggi in history: {len(history)}")

👤 Mi chiamo Marco e sono di Sassari.
🤖 Piacere di conoscerti, Marco! Sassari è una bellissima città della Sardegna, con una storia ricca e affascinante. 

C'è qualcosa di specifico su cui posso aiutarti? Che si tratti di informazioni su Sassari, domande generali, o altro ancora, sono qui per te! 😊

👤 Qual è la capitale della Sardegna?
🤖 La capitale della Sardegna è **Cagliari**. 

Si trova nella parte meridionale dell'isola ed è il capoluogo della regione. È una città importante dal punto di vista storico, culturale ed economico per tutta la Sardegna.

👤 Come mi chiamo?
🤖 Ti chiami **Marco** e sei di Sassari! 😊

📊 Messaggi in history: 6


In [ ]:
# Vediamo quanti token stiamo usando
from anthropic import Anthropic

count = client.messages.count_tokens(
    model="claude-haiku-4-5-20251001",
    messages=history
)
print(f"📊 Token nella history attuale: {count.input_tokens}")
print(f"💰 Costo stimato prossima chiamata: ${count.input_tokens / 1_000_000 * 1.0:.6f}")
print()
print("💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!")

📊 Token nella history attuale: 212
💰 Costo stimato prossima chiamata: $0.000212

💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!


---
## 2. Gestire la Context Window

Tre strategie per evitare che la history cresca all'infinito.

In [ ]:
# STRATEGIA 1: Truncation
MAX_MESSAGGI = 6  # massimo 3 turni (user + assistant)

def chat_con_troncamento(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})

    # Tronca se troppo lunga
    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]
        print(f"  ✂️  History troncata a {MAX_MESSAGGI} messaggi")

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test
history = []
for i in range(5):
    r = chat_con_troncamento(f"Messaggio numero {i+1}. Quanti messaggi ricordi?")
    print(f"Turno {i+1} | History: {len(history)} msg | Risposta: {r[:80]}...")

Turno 1 | History: 2 msg | Risposta: Ricordo solo questo messaggio che hai appena scritto. Non ho memoria dei messagg...
Turno 2 | History: 4 msg | Risposta: Ricordo 2 messaggi:

1. Il tuo primo messaggio dove chiedevi quanti messaggi ric...
Turno 3 | History: 6 msg | Risposta: Ricordo 3 messaggi:

1. Il tuo primo messaggio ("Messaggio numero 1. Quanti mess...
  ✂️  History troncata a 6 messaggi
Turno 4 | History: 7 msg | Risposta: Ricordo 4 messaggi:

1. Il tuo primo messaggio ("Messaggio numero 1. Quanti mess...
  ✂️  History troncata a 6 messaggi
Turno 5 | History: 7 msg | Risposta: Ricordo 5 messaggi:

1. Il tuo primo messaggio ("Messaggio numero 1. Quanti mess...


In [ ]:
# STREAMING — output token per token
print("🌊 Risposta in streaming:\n")

full_text = ""
with client.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": "Raccontami in 3 frasi cos'è il machine learning."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
        full_text += text

print(f"\n\n📊 Testo totale: {len(full_text)} caratteri")

🌊 Risposta in streaming:

# Machine Learning

Il machine learning è una branca dell'intelligenza artificiale che permette ai computer di imparare dai dati senza essere esplicitamente programmati per ogni singolo compito. Invece di seguire istruzioni fisse, gli algoritmi identificano automaticamente pattern e regole nei dati per migliorare le loro prestazioni con l'esperienza. Viene utilizzato in moltissime applicazioni quotidiane, dai suggerimenti di film su Netflix alla traduzione automatica dei testi.

📊 Testo totale: 482 caratteri


---
## 3. Memoria Persistente su JSON

Salviamo la history su file — il chatbot ricorda tra sessioni diverse.

In [ ]:
import json, os

MEMORY_FILE = "chat_history.json"

def carica_storia():
    """Carica la history dal file JSON. Restituisce lista vuota se non esiste."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    """Salva la history su file JSON."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")

def chat_persistente(messaggio, system=None):
    """Chatbot con memoria persistente tra sessioni."""
    history.append({"role": "user", "content": messaggio})

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 400,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})

    salva_storia(history)  # Salva dopo ogni messaggio
    return testo

print("✅ Funzioni di persistenza pronte!")

✅ Funzioni di persistenza pronte!


In [ ]:
# Simulazione sessione 1
print("=" * 50)
print("SESSIONE 1")
print("=" * 50)

history = carica_storia()  # Carica eventuali messaggi precedenti

r = chat_persistente("Ciao! Mi chiamo Luca e studio AI engineering a Sassari.")
print(f"🤖 {r}\n")

r = chat_persistente("Qual è la lezione più difficile secondo te?")
print(f"🤖 {r}\n")

print("\n--- Fine sessione 1 ---")
print(f"History salvata con {len(history)} messaggi")

SESSIONE 1
🆕 Nessuna storia precedente — nuova conversazione
💾 Storia salvata: 2 messaggi
🤖 Ciao Luca! 👋 Piacere di conoscerti!

Che interessante - studi AI engineering a Sassari! È un campo affascinante e in crescita. Mi piacerebbe saperne di più:

- Che anno sei?
- Quali aspetti dell'AI ti interessano di più? (machine learning, NLP, computer vision, ecc.)
- Ci sono progetti specifici su cui stai lavorando?

Sono qui se hai domande su argomenti di studio, progetti, o semplicemente se vuoi scambiare due chiacchiere! 😊

💾 Storia salvata: 4 messaggi
🤖 Buona domanda! Secondo me, le parti più impegnative dell'AI engineering di solito sono:

1. **Matematica sottostante** - Calcolo, algebra lineare, probabilità e statistiche. Non è solo memorizzare, ma *veramente capire* come e perché funzionano gli algoritmi.

2. **Il gap teoria-pratica** - Leggere di una rete neurale è diverso da implementarla bene. I dettagli implementativi (normalizzazione dati, hyperparameter tuning, debugging modelli) 

In [ ]:
# Simulazione sessione 2 — ricarica la memoria!
print("=" * 50)
print("SESSIONE 2 (nuova sessione, stessa memoria)")
print("=" * 50)

history = carica_storia()  # Ricarica dal file

r = chat_persistente("Come mi chiamo? Ricordi cosa stavo studiando?")
print(f"🤖 {r}")

SESSIONE 2 (nuova sessione, stessa memoria)
📂 Storia caricata: 4 messaggi precedenti
💾 Storia salvata: 6 messaggi
🤖 Certo! 😊

Ti chiami **Luca** e studi **AI engineering a Sassari**.

L'ho ricordato dalla nostra conversazione appena iniziata. Però voglio essere trasparente: ricordo solo quello che mi hai detto *in questa conversazione specifica*. Se chiudessimo la chat e ricominciassimo domani, non avrei memoria di te - ogni conversazione è indipendente per me.

È una limitazione importante da sapere se pensi di lavorare con me regolarmente su progetti! 📝

C'è qualcosa di specifico su cui vuoi che ti aiuti?


---
## ⭐ Esercizi

In [ ]:
NOME_STUDENTE = ""  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

### Esercizio 1 — Chatbot multi-turno base ★☆☆
Crea un chatbot con system prompt WiData che mantiene la history. Fai almeno 4 domande collegate e verifica che risponda in modo coerente con i messaggi precedenti.

In [ ]:
# ESERCIZIO 1
history = []
system_widata = """  # ← scrivi il system prompt WiData
"""

# Fai almeno 4 domande collegate
# domanda1 = chat("...", system=system_widata)
# ...

### Esercizio 2 — Sliding Window ★★☆
Implementa la sliding window: mantieni sempre il system prompt + gli ultimi 4 turni (8 messaggi). Testa che dopo 6 turni il chatbot non ricordi più i primi messaggi ma ricordi gli ultimi.

In [ ]:
# ESERCIZIO 2
MAX_TURNS = 4  # mantieni solo gli ultimi 4 turni

def chat_sliding_window(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})

    # TODO: implementa sliding window
    # Suggerimento: history[-MAX_TURNS*2:] mantiene gli ultimi N turni

    # risposta = client.messages.create(...)
    # history.append(...)
    # return ...
    pass

# Test: la prima informazione viene dimenticata dopo 4 turni?
history = []
# chat_sliding_window("Mi chiamo Alice")
# ... aggiungi altri messaggi ...
# chat_sliding_window("Come mi chiamo?")

### Esercizio 3 — Chatbot con streaming ★★☆
Riscrivi la funzione `chat()` usando lo streaming. L'output deve apparire parola per parola nel terminale. Aggiungi anche il conteggio dei token al termine.

In [ ]:
# ESERCIZIO 3
def chat_streaming(messaggio, history, system=None):
    history.append({"role": "user", "content": messaggio})

    # TODO: usa client.messages.stream() invece di client.messages.create()
    # Stampa ogni token con print(text, end='', flush=True)
    # Accumula il testo completo in una variabile
    # Aggiungi la risposta completa alla history
    # Alla fine stampa il numero di token usati

    pass

# Test
history = []
# chat_streaming("Spiegami RAG in 3 frasi", history)

### Esercizio 4 — Chatbot con memoria persistente ★★★ (Deliverable!)

Costruisci il chatbot completo `chatbot_cli.py` con:
- History multi-turno
- Sliding window (max 10 messaggi)
- Streaming
- Memoria su JSON persistente
- System prompt WiData
- Loop interattivo con `input()` (digita 'esci' per uscire)

In [ ]:
# ESERCIZIO 4 — Chatbot completo (DELIVERABLE)
# Scrivi qui il codice completo del chatbot

import anthropic, json, os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

MEMORY_FILE = "chatbot_widata.json"
MAX_MESSAGGI = 10

SYSTEM = """
# ← Scrivi qui il tuo system prompt WiData
"""

def carica_storia():
    pass  # ← implementa

def salva_storia(history):
    pass  # ← implementa

def chat(messaggio, history):
    pass  # ← implementa con streaming + sliding window

# Loop principale
def main():
    history = carica_storia()
    print("🤖 Chatbot WiData avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        risposta = chat(utente, history)
        # Lo streaming stampa già durante l'esecuzione

# main()  # Decommentare per eseguire

---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione3_TUONOME.ipynb`
4. Carica su GitHub in `lezione3/`

```bash
git add lezione3/
git commit -m "Lezione 3 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 28/05)
Leggi **Huyen Cap. 6 — sezione RAG**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*